# Week 9 · Who drives the story? Build it, one move at a time

**LDSCI6253 Information Presentation & Data Visualisation** · Northeastern University London

This is the lecture's code, the Build-it slides, to run and change. One chart of Marylebone Road's NO₂ grows into a
story: the finding in the title, notes where it happened, one road in colour and the rest grey, the reader's turn to
explore, a martini glass, and a guess before the answer.

| § | Build-it | The move | The advanced Altair |
|---|---|---|---|
| 1 | Start with the bare line | a correct chart that says nothing | `mark_line` |
| 2 | Say the finding in the title | **say it** | layers (`+`), `mark_rule`, `alt.Title` |
| 3 | Write on the chart where it happened | **point at it** | a notes table, `mark_rule` with `y2`, `mark_text` |
| 4 | One line in colour, the rest grey | **isolate it** | a context layer, `detail`, layer order |
| 5 | Let the reader drive | **let go** | `selection_point`, `tooltip`, a colour scale |
| 6 | A martini glass: chapters, then open | **lead, then let go** | `alt.param`, `transform_filter` |
| 7 | Guess first, then show | **make it felt** | sliders and checkboxes, `XDatum` |
| 8 | Three ways to lead, then let go | martini glass, slideshow, drill-down | a chapter parameter, a linked selection, `&` |
| 9 | Four ways to make a number land | guess, count, human scale, physical | unit charts: `transform_window`, `mark_square`, `facet` |

Run the cells **in order**: each section builds on the one before. Each ends with a **Try it**: change the code and
run it again. The lab challenge (`09_Activity_IPDV_Brixton_Road_Story.ipynb`) uses the same moves on another road.

## 0 · Set up

Put `no2_london.csv` beside this notebook (it is on Canvas); if it is not there, `load` reads it from the course site.

In [ ]:
import pandas as pd
from pathlib import Path

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week9/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

## 1 · Start with the bare line

Correct, but silent: what should a reader see? The first lines are the setup every later section uses.

In [ ]:
import altair as alt
import pandas as pd

air = load('no2_london')
road = air[air['type'].isin(['Roadside', 'Kerbside'])]
my1 = air[air['code'] == 'MY1']      # Marylebone Road

alt.Chart(my1).mark_line(point=True).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
).properties(width=560, height=300)

**Try it.** Change `my1` to Brixton Road (code `'LB4'`): `air[air['code'] == 'LB4']`. What is the same, and what does the chart still not say?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
brixton = air[air['code'] == 'LB4']
alt.Chart(brixton).mark_line(point=True).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
).properties(width=560, height=300)
```
</details>

## 2 · Say the finding in the title

**Say it.** A title that is a finding, a subtitle that says how to read it, and the legal limit as a second layer: `line + limit` draws one chart on top of the other.

In [ ]:
red = '#c8102e'
line = alt.Chart(my1).mark_line(strokeWidth=3, color=red).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
)
limit = alt.Chart().mark_rule(strokeDash=[6, 4]).encode(
    y=alt.datum(40))

finding = ('Marylebone Road’s air has a third'
           ' of the NO₂ it had in 2010')
how = 'Yearly average; dashed: the legal limit, 40 µg/m³'

(line + limit).properties(
    width=560, height=300,
    title=alt.Title(finding, subtitle=how),
)

**Try it.** Add the WHO guideline (10 µg/m³) as a dotted third layer: `alt.Chart().mark_rule(strokeDash=[2, 3]).encode(y=alt.datum(10))`.

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
who = alt.Chart().mark_rule(strokeDash=[2, 3]).encode(y=alt.datum(10))
(line + limit + who).properties(width=560, height=300,
    title=alt.Title(finding, subtitle=how))
```
</details>

## 3 · Write on the chart where it happened

**Point at it.** A small table of notes: each note is a year, a height and its words. A rule drops from each note to the axis; the words sit beside it.

In [ ]:
# line, limit, finding, how: as on the last slide
notes = pd.DataFrame({
    'year': [2019, 2020, 2021, 2023],
    'at':   [172, 157, 142, 127],     # how high each sits
    'text': ['Central ULEZ', 'Lockdown',
             'ULEZ reaches here', 'London-wide'],
})
rules = alt.Chart(notes).mark_rule(color='grey').encode(
    x='year:O', y='at:Q', y2=alt.datum(0))  # up to its note
words = alt.Chart(notes).mark_text(align='left', dx=5).encode(
    x='year:O', y='at:Q', text='text:N')

(line + limit + rules + words).properties(
    width=560, height=300,
    title=alt.Title(finding, subtitle=how),
)

**Try it.** Add a fifth note: 2010, height 110, 'Start of the record'. Add a row to each list in `notes`, then run the cell again.

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
notes = pd.DataFrame({
    'year': [2010, 2019, 2020, 2021, 2023],
    'at':   [110, 172, 157, 142, 127],
    'text': ['Start of the record', 'Central ULEZ', 'Lockdown',
             'ULEZ reaches here', 'London-wide'],
})
# then run the rules, words and chart lines of this section again
```
</details>

## 4 · One line in colour, the rest grey

**Isolate it.** Every other roadside monitor as thin grey lines (`detail` draws one line per monitor, all the same colour), then Marylebone Road on top in red. The layer added last is drawn on top.

In [ ]:
# line, limit, notes, finding, how: as before
grey = '#a6a6a6'
others = alt.Chart(road).mark_line(color=grey, strokeWidth=1)
others = others.encode(
    x='year:O', y='no2:Q',
    detail='code:N',                 # one line per monitor
)

# the layer added last is drawn on top: grey, then red
(others + line + limit + rules + words).properties(
    width=560, height=300,
    title=alt.Title(finding, subtitle=how),
)

**Try it.** Put `others` last instead of first: `(line + limit + rules + words + others)`. What happens to the red line, and why?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
(line + limit + rules + words + others).properties(width=560, height=300,
    title=alt.Title(finding, subtitle=how))
# The grey lines are drawn over the red one: the story is buried. Order is meaning.
```
</details>

## 5 · Let the reader drive: tooltip and legend

**Let go.** Every monitor, coloured by type; click a type in the legend to keep it bright; point at a line for its site and value. The reader finds their own road, but nothing tells them the finding.

In [ ]:
# after the setup: air, road and my1
pick = alt.selection_point(fields=['type'], bind='legend')
# colours a colour-blind reader can tell apart (Okabe-Ito)
types = ['Kerbside', 'Roadside', 'Urban Background',
         'Suburban', 'Industrial']
colours = ['#d55e00', '#e69f00', '#0072b2',
           '#56b4e9', '#8c8c8c']

alt.Chart(air).mark_line(point=True).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
    detail='code:N',
    color=alt.Color('type:N', title='Monitor',
        scale=alt.Scale(domain=types, range=colours)),
    opacity=alt.when(pick).then(alt.value(0.9))
                          .otherwise(alt.value(0.05)),
    tooltip=['site:N', 'borough:N', 'type:N',
             'year:O', 'no2:Q'],
).add_params(pick).properties(width=560, height=300)

**Try it.** Add `'code:N'` to the tooltip list, then find Brixton Road (LB4) by pointing at the lines.

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
# in the encode(...) of this section:
tooltip=['site:N', 'borough:N', 'type:N', 'code:N',
         'year:O', 'no2:Q'],
```
</details>

## 6 · A martini glass: chapters, then open

**Lead, then let go.** A parameter bound to three buttons is the chapter. Chapter 1 is the finding alone; 2 adds the notes (the stem); 3 opens the bowl: every road, to explore. The axis stays still, so nothing jumps between chapters.

In [ ]:
# line, limit, rules, words, others, finding, how: as before
chapter = alt.param(value=1, bind=alt.binding_radio(
    options=[1, 2, 3], name='Chapter '))

told = (rules.transform_filter(chapter >= 2)   # the stem
        + words.transform_filter(chapter >= 2))
dots = alt.OverlayMarkDef(color=grey, size=15)
bowl = others.mark_line(color=grey, point=dots).encode(
    tooltip=['site:N', 'borough:N', 'year:O', 'no2:Q'],
).transform_filter(chapter >= 3)               # the bowl

still = alt.Scale(domain=[0, 180])    # the axis stays still
hero = line.encode(y=alt.Y('no2:Q', scale=still,
                           title='NO₂ (µg/m³)'))

(bowl + hero + limit + told).add_params(chapter).properties(
    width=560, height=300,
    title=alt.Title(finding, subtitle=how),
)

**Try it.** Make it open on chapter 3, the bowl, by changing `value=1` to `value=3`. Which do you think a hurried reader should see first, and why?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
chapter = alt.param(value=3, bind=alt.binding_radio(
    options=[1, 2, 3], name='Chapter '))
# A hurried reader should get chapter 1: the finding first, then the bowl.
```
</details>

## 7 · Guess first, then show

**Make it felt.** The reader moves a slider to guess 2025, then ticks *Show me*. Guessing first makes the answer stick (the lecture's *You Draw It*).

In [ ]:
# after the setup: air, road and my1
guess = alt.param(value=60, bind=alt.binding_range(
    min=0, max=100, step=1, name='Your guess for 2025 '))
show = alt.param(value=False,
                 bind=alt.binding_checkbox(name='Show me '))

years = list(range(2010, 2026))   # room for the years to guess
base = alt.Chart(my1).encode(
    x=alt.X('year:O', title=None,
            scale=alt.Scale(domain=years)),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)',
            scale=alt.Scale(domain=[0, 100])))
known = base.mark_line(strokeWidth=3, color='#0c3354')
known = known.transform_filter(alt.datum.year <= 2016)
truth = base.mark_line(strokeWidth=6, color='#c8102e')
truth = truth.transform_filter(
    (alt.datum.year >= 2016) & show)
yours = alt.Chart().mark_point(size=260, filled=True,
                               color='#0c3354').encode(
    x=alt.XDatum(2025, type='ordinal'),
    y=alt.YDatum(guess, type='quantitative'))

(known + truth + yours).add_params(guess, show).properties(
    width=560, height=300,
    title='Marylebone Road, 2016: 89. Guess 2025.')

**Try it.** Start the slider at 30 instead of 60 (`value=30`). Does a different starting guess change what people guess? (It does: the anchor.)

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
guess = alt.param(value=30, bind=alt.binding_range(
    min=0, max=100, step=1, name='Your guess for 2025 '))
```
</details>

## 8 · Three ways to lead, then let go

The lecture's slide *Three ways to lead, then let go* (Segel and Heer, 2010) names three shapes for a story that
starts with the author and ends with the reader:

| Shape | The author leads by… | then the reader… | Here |
|---|---|---|---|
| **Martini glass** | a short story in order (the stem) | explores everything (the bowl) | §6 |
| **Interactive slideshow** | chapters in order | explores inside each chapter | 8a |
| **Drill-down story** | one overview | picks which story to open, and the author tells it | 8b |

### 8a · An interactive slideshow

The author picks the chapters: three years, each with its own caption. Inside a chapter the reader can point at any
monitor. The captions are worked out from the data, so they can never say a number the chart does not show.

In [ ]:
# after §5: air, types and colours
picked = [2010, 2019, 2025]                       # the author's chapters
said = []
for y in picked:
    that = air[air['year'] == y]
    said.append(f"{y}: {(that['no2'] > 40).sum()} of {len(that)} monitors over the limit")
captions = pd.DataFrame({'year': picked, 'say': said})

year = alt.param(value=2010, bind=alt.binding_radio(options=picked, name='Chapter '))

dots = alt.Chart(air).mark_circle(size=90, opacity=0.8).encode(
    x=alt.X('no2:Q', title='NO₂ (µg/m³), yearly average', scale=alt.Scale(domain=[0, 180])),
    y=alt.Y('type:N', title=None, sort=types),
    color=alt.Color('type:N', scale=alt.Scale(domain=types, range=colours), legend=None),
    tooltip=['site:N', 'borough:N', 'no2:Q'],      # the reader explores inside the chapter
).transform_filter(alt.datum.year == year)
caption = alt.Chart(captions).mark_text(align='left', fontSize=16, fontWeight='bold').encode(
    x=alt.value(0), y=alt.value(-12), text='say:N').transform_filter(alt.datum.year == year)
limit_x = alt.Chart().mark_rule(strokeDash=[6, 4]).encode(x=alt.datum(40))

(dots + limit_x + caption).add_params(year).properties(width=560, height=240)

**Try it.** Add a fourth chapter, 2021, to `picked` and to the buttons. What does its caption say?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
picked = [2010, 2019, 2021, 2025]
# then run the cell again: the captions and the buttons both come from picked
```
</details>

### 8b · A drill-down story

One overview: the median NO₂ for each type of monitor, year by year. **Click a line** to open its story below: every
monitor of that type, to point at. The author still leads: the overview is drawn first, and the reader opens one
story at a time.

In [ ]:
# after §5: air, types and colours
opened = alt.selection_point(fields=['type'], value='Kerbside')   # opens on the kerbside story

overview = alt.Chart(air).mark_line(strokeWidth=4, point=True).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('median(no2):Q', title='Median NO₂ (µg/m³)'),
    color=alt.Color('type:N', scale=alt.Scale(domain=types, range=colours), title='Monitor (click one)'),
    opacity=alt.when(opened).then(alt.value(1)).otherwise(alt.value(0.25)),
).add_params(opened).properties(width=560, height=200, title='Every type of monitor fell')

story = alt.Chart(air).mark_line(color='#0c3354', strokeWidth=1.5).encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('no2:Q', title='NO₂ (µg/m³)'),
    detail='code:N',
    tooltip=['site:N', 'borough:N', 'year:O', 'no2:Q'],
).transform_filter(opened).properties(width=560, height=200, title='The type you opened, monitor by monitor')

overview & story                                  # & puts one chart above the other

**Try it.** Make the drill-down open on Urban Background instead: change `value='Kerbside'`. Which type's story
would you tell a resident who lives on a quiet street?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
opened = alt.selection_point(fields=['type'], value='Urban Background')
# A quiet street is nearer an urban background monitor than a kerbside one.
```
</details>

## 9 · Four ways to make a number land

The lecture's slide *Four ways to make a number land*. A number is read; these make it **felt** (Kennedy and Hill,
2018).

| Way | How | Here |
|---|---|---|
| **Guess first** | the reader guesses, then the truth appears beside the guess | §7 |
| **Count it out** | one square per thing | 9a |
| **Human scale** | in units people know | 9b |
| **Make it physical** | stack it, print it, build it | 9c |

### 9a · Count it out

A concentration is not a count, so count something that is: **monitors**. One square each, red if it was over the
legal limit that year. `transform_window` numbers the squares; `transform_calculate` lays them out in rows of ten.

In [ ]:
# after §2: red
counted = air[air['year'].isin([2010, 2025])].copy()
counted['over'] = counted['no2'] > 40

squares = alt.Chart(counted).transform_window(
    i='row_number()', groupby=['year'], sort=[alt.SortField('no2', order='descending')]
).transform_calculate(
    col='(datum.i - 1) % 10', row='floor((datum.i - 1) / 10)'
).mark_square(size=260).encode(
    x=alt.X('col:O', axis=None),
    y=alt.Y('row:O', axis=None),
    color=alt.when(alt.datum.over).then(alt.value(red)).otherwise(alt.value('#c9d3dd')),
    tooltip=['site:N', 'no2:Q'],
).properties(width=200, height=200)

squares.facet(column=alt.Column('year:O', title=None)).properties(
    title='Red: a monitor over the legal limit. One square per monitor.')

**Try it.** Count 2019 as well: add it to the list in the first line. How many squares are red?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
counted = air[air['year'].isin([2010, 2019, 2025])].copy()
counted['over'] = counted['no2'] > 40
# then run the rest of the cell again
```
</details>

### 9b · Human scale

"98 µg/m³" means little to most readers. The WHO's guideline for NO₂ (a yearly average of 10 µg/m³, 2021) is a
unit: **one block, one guideline's worth**. Marylebone Road's air held about ten in 2010, and about three now.

In [ ]:
who = 10                                          # the WHO guideline, µg/m³
pair = air[(air['code'] == 'MY1') & air['year'].isin([2010, 2025])]
blocks = pd.DataFrame([{'year': r.year, 'block': k}
                       for r in pair.itertuples() for k in range(round(r.no2 / who))])

alt.Chart(blocks).mark_square(size=600, color='#0c3354').encode(
    x=alt.X('block:O', axis=None),
    y=alt.Y('year:O', title=None, sort='descending'),
).properties(width=420, height=120,
             title='Marylebone Road: one block is the WHO guideline (10 µg/m³)')

**Try it.** Use the legal limit (40) as the unit instead. How many blocks now, and which unit tells a reader more?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
who = 40
# 2010: about 2 blocks; 2025: about 1. The legal limit makes 2025 look fine;
# the WHO unit shows the air is still three times what is thought safe.
```
</details>

### 9c · Make it physical

The same blocks, stacked into towers, to print, cut out and stand on a desk: a thing you can hold or walk around
(Jansen et al., 2015, call this *data physicalisation*). Save the chart as a file and print it.

In [ ]:
tower = alt.Chart(blocks).mark_square(size=900, color=red, stroke='white', strokeWidth=3).encode(
    x=alt.X('year:O', title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('block:O', axis=None, sort='descending'),
).properties(width=200, height=360, title='Stack them: 2010 against 2025')
# tower.save('marylebone_tower.html')             # open it in a browser and print it
tower

**Try it.** Build the tower for Brixton Road (code `'LB4'`) next to Marylebone Road's. Which would you rather
live on?

<details><summary><b>Stuck? Open a model answer</b></summary>

```python
pair = air[air['code'].isin(['MY1', 'LB4']) & air['year'].isin([2010, 2025])]
blocks = pd.DataFrame([{'road': r.site, 'year': r.year, 'block': k}
                       for r in pair.itertuples() for k in range(round(r.no2 / who))])
alt.Chart(blocks).mark_square(size=500, color=red, stroke='white', strokeWidth=3).encode(
    x=alt.X('year:O', title=None), y=alt.Y('block:O', axis=None, sort='descending'),
).facet(column='road:N')
```
</details>

## Next: the lab

Open `09_Activity_IPDV_Brixton_Road_Story.ipynb`. Brixton Road's story is harder: it fell by about 72%, and it is
still London's worst road. You will need §4's move, isolating it from every other road, and two new ones: a shaded
band, and one chart whose colour changes by a condition (`alt.when`).